# client

> Thin HTTP client for one `estravon-backend` instance, plus local-subprocess
> orchestration -- one process per engine, since each running `estravon-backend`
> instance is pinned to a single engine (see its own docs) and this package is
> what drives several of them side by side for a comparison.

In [ ]:
#| default_exp client

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import subprocess
import tempfile
import time
from pathlib import Path

import httpx

## Client

`estravon-backend` is an HTTP server: it listens on a port, and we talk to
it by sending web requests, the same way a browser talks to a website. A
`Client` is a small object that does that talking for us -- we create one
pointed at a server's address (`Client("http://localhost:7766")`), then call
plain Python methods on it (`submit_and_wait(...)`) instead of constructing
HTTP requests by hand.

`estravon-backend` can respond to a submitted PDF in one of two ways, and
we need to handle both:

- **It replies right away with the finished result.** This is what happens
  when we're running a copy of the backend on our own machine: it does
  the whole extraction before it answers, so the reply -- an HTTP status
  code of **200** -- already contains everything we asked for.
- **It replies immediately with "received, come back later."** This is what
  happens with Estravon's own paid hosted service: jobs go into a queue and
  may take a while, so the first reply is just an acknowledgement -- HTTP
  status **202** -- with a job id, and we have to ask again and again
  ("is it done yet?") until the answer is finally "done" or "error." This
  repeated asking is called *polling*.

Without a `Client`, code that wants to work against either kind of server
has to check "did we get a 200 or a 202?" every single time and handle both
branches itself. `Client.submit_and_wait()` does that checking once, inside
itself, and always hands us back a finished result -- so the rest of
this package (and anyone using it) never has to think about which of the
two cases just happened.

There is a second wrinkle, independent of 200-vs-202: even a finished result
does not contain the extracted text itself, only a *link* to it (`md_url`).
Getting the actual Markdown is a second, separate request. This is spelled
out in `estravon-backend`'s own `docs/API.md`, under "The Markdown text is
NOT in the `/process` response" -- worth reading once, because it is the
single most common mistake when calling this API directly. `Client` hides
this too: `fetch_markdown()` does that second request for us.

In [ ]:
#| export
class Client:
    """Talks to ONE running estravon-backend instance over HTTP.

    Handles both response shapes transparently:
    - local server: `POST /process` returns `200` with the full result inline
      (still requires the two-step `/files/...` fetch for the Markdown text).
    - hosted server: `POST /process` returns `202 {"job_id":...}`, poll
      `GET /jobs/{job_id}` until `status` is `"done"`/`"error"`.
    """

    def __init__(
        self,
        base_url: str,
        api_key: str | None = None,
        timeout_s: float = 30.0,
        transport: httpx.BaseTransport | None = None,
    ):
        """`transport` is normally left as None (real network) -- overriding it
        with an `httpx.MockTransport` is how tests exercise this class without
        a live server, since `httpx.get`/`httpx.post` module-level functions
        build their own internal client and can't be monkeypatched from outside."""
        self.base_url = base_url.rstrip("/")
        self.api_key = api_key
        self.timeout_s = timeout_s
        self._http = httpx.Client(transport=transport, timeout=timeout_s)

    def _headers(self) -> dict[str, str]:
        return {"X-API-Key": self.api_key} if self.api_key else {}

    def ping(self) -> dict:
        r = self._http.get(f"{self.base_url}/ping", headers=self._headers())
        r.raise_for_status()
        return r.json()

    def submit(
        self,
        pdf_path: str,
        section_name: str,
        page_range: str,
        chunk_size: int = 80,
        mode: str = "balanced",
        force_ocr: bool = False,
    ) -> dict:
        """POST /process. Returns the raw JSON body (either a done result or a queued ack)."""
        pdf_bytes = Path(pdf_path).read_bytes()
        r = self._http.post(
            f"{self.base_url}/process",
            headers=self._headers(),
            files={"pdf_file": (Path(pdf_path).name, pdf_bytes, "application/pdf")},
            data={
                "section_name": section_name,
                "page_range": page_range,
                "chunk_size": str(chunk_size),
                "mode": mode,
                "force_ocr": "true" if force_ocr else "false",
            },
        )
        body = r.json()
        if r.status_code not in (200, 202):
            raise RuntimeError(f"submit failed ({r.status_code}): {body.get('error', body)}")
        return body

    def poll_until_done(self, job_id: str, max_wait_s: float = 300.0, interval_s: float = 3.0) -> dict:
        """Poll GET /jobs/{job_id} until status is done/error. Hosted-only -- the
        local server has no job queue, so callers only reach this after a 202."""
        deadline = time.monotonic() + max_wait_s
        while time.monotonic() < deadline:
            r = self._http.get(f"{self.base_url}/jobs/{job_id}", headers=self._headers())
            body = r.json()
            if body.get("status") in ("done", "error", "failed"):
                return body
            time.sleep(interval_s)
        return {"status": "timeout", "job_id": job_id}

    def submit_and_wait(
        self, *args, max_wait_s: float = 300.0, interval_s: float = 3.0, **kwargs
    ) -> dict:
        """submit(), then poll only if the response was a 202 queued ack.

        A local instance's submit() already returns the finished result (200),
        so this is a no-op pass-through in that case -- callers never need to
        branch on local-vs-hosted themselves.
        """
        result = self.submit(*args, **kwargs)
        if result.get("status") == "queued" and result.get("job_id"):
            return self.poll_until_done(result["job_id"], max_wait_s=max_wait_s, interval_s=interval_s)
        return result

    def fetch_text(self, path: str) -> str:
        """GET a /files/{job_id}/{filename} path. `path` is the exact value
        from a result's `files[].md_url` -- the two-step fetch, step two."""
        r = self._http.get(f"{self.base_url}{path}", headers=self._headers())
        r.raise_for_status()
        return r.text

    def fetch_markdown(self, result: dict) -> list[dict]:
        """Fetch the Markdown text for every file in a done `/process` result.

        Returns [{"label", "markdown"}, ...] -- the actual text, not just URLs.
        """
        out = []
        for f in result.get("files", []):
            out.append({"label": f["label"], "markdown": self.fetch_text(f["md_url"])})
        return out

    def close(self) -> None:
        self._http.close()

### Try it: the 200 case (local server, answer arrives immediately)

There is no real server running in this notebook, so the two cells below
build a *fake* one: a small function that looks at the incoming request and
decides what a real `estravon-backend` would have said back, without
actually running one. This is the standard way to test code that talks over
HTTP without needing a live server or network access -- `httpx` (the library
`Client` is built on) calls this a "mock transport."

This first fake server always answers a submission immediately with a
finished result -- the 200 case. The point of running this is to see
`submit_and_wait()` recognise a 200 and hand the result straight back, and
`fetch_markdown()` then make the second request to turn `md_url` into actual
text.

In [ ]:
#| hide
import httpx as _httpx


def _fake_local_transport(request: _httpx.Request) -> _httpx.Response:
    if request.url.path == "/process":
        return _httpx.Response(200, json={
            "status": "done", "backend": "mineru", "job_id": "job1",
            "files": [{"label": "ch", "md_url": "/files/job1/ch.md", "image_urls": [], "quality_score": None}],
            "page_count": 3, "total_predict_time_s": 1.2, "total_cost_usd": 0.0, "local": True,
        })
    if request.url.path == "/files/job1/ch.md":
        return _httpx.Response(200, text="# Hello from mineru")
    raise AssertionError(f"unexpected path {request.url.path}")


import tempfile as _tf
with _tf.NamedTemporaryFile(suffix=".pdf") as _f:
    _f.write(b"%PDF-1.4 stub")
    _f.flush()
    c = Client("http://fake", transport=_httpx.MockTransport(_fake_local_transport))
    result = c.submit_and_wait(_f.name, "ch", "1-3")
    assert result["status"] == "done"
    md = c.fetch_markdown(result)
    assert md[0]["markdown"] == "# Hello from mineru"
    c.close()
print("submit_and_wait() returned:", result)
print("fetch_markdown() returned:", md)

### Try it: the 202 case (hosted server, answer arrives after polling)

This second fake server behaves like Estravon's hosted service instead: the
first request gets a 202 with a job id and no result yet. The fake server is
written to answer "still queued" the first time it's asked about that job,
then "done" the second time -- so the cell below can show the actual back
and forth: submit, get told to wait, ask again, get the real answer.

`submit_and_wait()` is the same method called the same way as in the 200
case above -- the caller does not need to know in advance which of the two
this particular server will do.

In [ ]:
#| hide
_poll_count = 0


def _fake_hosted_transport(request: _httpx.Request) -> _httpx.Response:
    global _poll_count
    if request.url.path == "/process":
        return _httpx.Response(202, json={"status": "queued", "job_id": "job2"})
    if request.url.path == "/jobs/job2":
        _poll_count += 1
        if _poll_count == 1:
            print("  poll 1: still queued")
            return _httpx.Response(200, json={"status": "queued", "job_id": "job2"})
        print("  poll 2: done")
        return _httpx.Response(200, json={
            "status": "done", "backend": "mistral", "job_id": "job2",
            "files": [{"label": "ch", "md_url": "/files/job2/ch.md", "image_urls": [], "quality_score": None}],
            "page_count": 3, "total_predict_time_s": 2.4, "total_cost_usd": 0.006, "local": False,
        })
    if request.url.path == "/files/job2/ch.md":
        return _httpx.Response(200, text="# Hello from mistral")
    raise AssertionError(f"unexpected path {request.url.path}")


with _tf.NamedTemporaryFile(suffix=".pdf") as _f:
    _f.write(b"%PDF-1.4 stub")
    _f.flush()
    c2 = Client("http://fake", transport=_httpx.MockTransport(_fake_hosted_transport))
    print("submitting...")
    result2 = c2.submit_and_wait(_f.name, "ch", "1-3", max_wait_s=5, interval_s=0.01)
    assert result2["status"] == "done"
    assert _poll_count == 2
    md2 = c2.fetch_markdown(result2)
    assert md2[0]["markdown"] == "# Hello from mistral"
    c2.close()
print("submit_and_wait() returned:", result2)
print("fetch_markdown() returned:", md2)

## Running a local backend for ourselves, one engine at a time

Comparing engines means talking to several `estravon-backend` instances, and
each running instance is pinned to a single engine -- there is no way to ask
one server "use MinerU this time, Mistral next time" (see
`estravon-backend`'s own docs for why). So comparing N engines means N
servers running at once, each started with a different `--backend` flag and
a different port.

`LocalEngineProcess` is what starts one of those servers for us, in the
background, and cleans it up afterward -- the equivalent of opening a
terminal, typing `estravon --backend mistral --port 7767`, waiting for the
"ready" message, and later pressing Ctrl-C when we're done, except done
automatically and safely from Python. Used as:

```python
with LocalEngineProcess("mistral", 7767) as proc:
    ...   # proc.base_url is now reachable
# the background server is stopped here, even if the code above raised
```

Internally, entering the `with` block runs the `estravon` command as a
subprocess and repeatedly checks `/ping` until it answers (or gives up after
`ready_timeout_s` seconds). Leaving the block -- normally or via an error --
asks the subprocess to stop, and force-kills it if it doesn't within a few
seconds, so a crashed comparison never leaves a stray server running in the
background. From here on, starting a server this way is called **Mode A**,
as opposed to **Mode B**: pointing at servers someone else already started
elsewhere, with no `LocalEngineProcess` involved. `compare()`, in
`03_compare`, is where a caller picks between the two.

In [ ]:
#| export
class LocalEngineProcess:
    """Context manager: spawn `estravon --backend <engine> --port <port>`,
    wait for /ping, tear down on exit."""

    def __init__(self, engine: str, port: int, ready_timeout_s: float = 60.0, estravon_cmd: str = "estravon"):
        self.engine = engine
        self.port = port
        self.ready_timeout_s = ready_timeout_s
        self.estravon_cmd = estravon_cmd
        self._proc: subprocess.Popen | None = None
        self._job_dir: tempfile.TemporaryDirectory | None = None

    @property
    def base_url(self) -> str:
        return f"http://127.0.0.1:{self.port}"

    def __enter__(self) -> "LocalEngineProcess":
        self._job_dir = tempfile.TemporaryDirectory(prefix=f"estravon-bench-{self.engine}-")
        self._proc = subprocess.Popen(
            [self.estravon_cmd, "--backend", self.engine, "--port", str(self.port),
             "--job-dir", self._job_dir.name],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        )
        deadline = time.monotonic() + self.ready_timeout_s
        last_err: Exception | None = None
        while time.monotonic() < deadline:
            if self._proc.poll() is not None:
                out = self._proc.stdout.read() if self._proc.stdout else ""
                raise RuntimeError(f"estravon --backend {self.engine} exited early:\n{out}")
            try:
                httpx.get(f"{self.base_url}/ping", timeout=2.0).raise_for_status()
                return self
            except Exception as exc:
                last_err = exc
                time.sleep(1.0)
        self._teardown()
        raise TimeoutError(f"{self.engine} instance did not become ready within {self.ready_timeout_s}s: {last_err}")

    def __exit__(self, *exc_info) -> None:
        self._teardown()

    def _teardown(self) -> None:
        if self._proc is not None and self._proc.poll() is None:
            self._proc.terminate()
            try:
                self._proc.wait(timeout=10)
            except subprocess.TimeoutExpired:
                self._proc.kill()
        if self._job_dir is not None:
            self._job_dir.cleanup()

In [ ]:
#| hide
# LocalEngineProcess needs a real `estravon` on PATH to fully exercise -- not
# available in this test environment, so this only checks the teardown-without-
# start path (constructing and tearing down without ever entering) is inert.
p = LocalEngineProcess("mineru", 9999)
p._teardown()  # no-op: nothing was ever started
assert p._proc is None
print(f"engine={p.engine!r} port={p.port} proc={p._proc}")

---
Next: [`02_report`](02_report.ipynb) -- `ComparisonResultList`, what we get
back once several of these `Client` calls have run and we want to look at
the results side by side.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()